# 06 — Preservação dos índices ecoacústicos

Apresenta a configuração congelada e calcula erros e correlações pareadas por índice, corpus, codec e bitrate.

> Execute as células de cima para baixo. Parâmetros editáveis aparecem em uma célula própria e todo artefato gerado informa seu destino.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    ROOT = ROOT.parent
if not (ROOT / "data" / "freeze-spec-v1.json").is_file():
    raise RuntimeError("Execute este notebook dentro do repositório CodEco.")

for location in (ROOT / "src", ROOT / "scripts"):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))

REPORTS = ROOT / "reports"
DERIVED = ROOT / "data" / "derived"
REPORTS.mkdir(exist_ok=True)
DERIVED.mkdir(exist_ok=True)
print(f"Repositório: {ROOT}")


In [ ]:
from pathlib import Path
print((ROOT / "configs" / "acoustic-indices-v1.yaml").read_text())


In [ ]:
import pandas as pd
import config as C
results_path = C.OUT_DIR / "results.csv"
results = pd.read_csv(results_path) if results_path.is_file() else pd.DataFrame()
index_columns = [column for column in results.columns if column.startswith("idx_")]
print("Índices disponíveis:", index_columns)


In [ ]:
if not results.empty and index_columns:
    from analyze import ccc
    from scipy.stats import pearsonr, spearmanr
    import numpy as np

    baseline = results[results.codec == "baseline"][["dataset", "clip_id", *index_columns]]
    paired = results[results.codec != "baseline"].merge(baseline, on=["dataset", "clip_id"], suffixes=("", "_reference"))
    rows = []
    for keys, group in paired.groupby(["dataset", "codec", "setting"]):
        for index in index_columns:
            a = group[f"{index}_reference"].to_numpy(float); b = group[index].to_numpy(float)
            ok = np.isfinite(a) & np.isfinite(b)
            if ok.sum() > 3 and a[ok].std() > 0 and b[ok].std() > 0:
                rows.append({"dataset": keys[0], "codec": keys[1], "setting": keys[2], "index": index,
                             "signed_error": float(np.mean(b[ok] - a[ok])),
                             "absolute_error": float(np.mean(np.abs(b[ok] - a[ok]))),
                             "pearson": pearsonr(a[ok], b[ok]).statistic,
                             "spearman": spearmanr(a[ok], b[ok]).statistic,
                             "ccc": ccc(a[ok], b[ok])})
    index_summary = pd.DataFrame(rows)
    index_summary.to_csv(REPORTS / "index_preservation.csv", index=False)
    display(index_summary)
else:
    print("Execute primeiro a avaliação do notebook 05.")
